In [3]:
import pandas as pd
import duckdb

data = [
    # Device A
    ["A", "2026-07-01 08:00:00", 70],
    ["A", "2026-07-01 08:05:00", 75],
    ["A", "2026-07-01 08:10:00", 68],
    ["A", "2026-07-01 08:15:00", 80],

    # Device B
    ["B", "2026-07-01 08:00:00", 60],
    ["B", "2026-07-01 08:05:00", 65],
    ["B", "2026-07-01 08:10:00", 72],

    # Device C
    ["C", "2026-07-01 08:00:00", 90],
    ["C", "2026-07-01 08:05:00", 85],
    ["C", "2026-07-01 08:10:00", 88],
]

df = pd.DataFrame(
    data,
    columns=["device_id", "collect_time", "temp_value"]
)

df["collect_time"] = pd.to_datetime(df["collect_time"])

print(df)



  device_id        collect_time  temp_value
0         A 2026-07-01 08:00:00          70
1         A 2026-07-01 08:05:00          75
2         A 2026-07-01 08:10:00          68
3         A 2026-07-01 08:15:00          80
4         B 2026-07-01 08:00:00          60
5         B 2026-07-01 08:05:00          65
6         B 2026-07-01 08:10:00          72
7         C 2026-07-01 08:00:00          90
8         C 2026-07-01 08:05:00          85
9         C 2026-07-01 08:10:00          88


## 题目要求

### 分别使用 SQL 和 Pandas 完成：

- 计算每个设备当前记录与上一条记录的温度差。

### 最终输出字段：

- `device_id`
- `collect_time`
- `temp_value`
- `previous_temp_value`
- `temp_diff`

In [9]:
# SQL轨道

query = """
WITH preview_temp AS (
SELECT
    device_id,
    collect_time,
    temp_value,
    LAG(temp_value,1) OVER(
        PARTITION BY device_id 
        ORDER BY collect_time
        ) AS previous_temp_value
FROM df
),
differ_table AS (
SELECT
    device_id,
    collect_time,
    temp_value,
    previous_temp_value,
    (temp_value - previous_temp_value) AS temp_diff
FROM preview_temp

)
SELECT *
FROM differ_table
ORDER BY device_id,collect_time
"""
df_sql = duckdb.execute(query).fetchdf()
df_sql

,device_id,collect_time,temp_value,previous_temp_value,temp_diff
0,A,2026-07-01 08:00:00,70,<NA>,<NA>
1,A,2026-07-01 08:05:00,75,70,5
2,A,2026-07-01 08:10:00,68,75,-7
3,A,2026-07-01 08:15:00,80,68,12
4,B,2026-07-01 08:00:00,60,<NA>,<NA>
5,B,2026-07-01 08:05:00,65,60,5
6,B,2026-07-01 08:10:00,72,65,7
7,C,2026-07-01 08:00:00,90,<NA>,<NA>
8,C,2026-07-01 08:05:00,85,90,-5
9,C,2026-07-01 08:10:00,88,85,3


In [10]:
# PANDAS轨道

df_pd = (
    df
    .sort_values(by=['device_id','collect_time'])
    .assign(
        previous_temp_value = lambda x:(
            x.groupby('device_id')['temp_value']
            .shift(1)
        ),
        temp_diff = lambda x:(
            x['temp_value'] - x['previous_temp_value']
        )
    )
    .reset_index(drop=True)
)
df_pd

,device_id,collect_time,temp_value,previous_temp_value,temp_diff
0,A,2026-07-01 08:00:00,70,NaN,NaN
1,A,2026-07-01 08:05:00,75,70.0,5.0
2,A,2026-07-01 08:10:00,68,75.0,-7.0
3,A,2026-07-01 08:15:00,80,68.0,12.0
4,B,2026-07-01 08:00:00,60,NaN,NaN
5,B,2026-07-01 08:05:00,65,60.0,5.0
6,B,2026-07-01 08:10:00,72,65.0,7.0
7,C,2026-07-01 08:00:00,90,NaN,NaN
8,C,2026-07-01 08:05:00,85,90.0,-5.0
9,C,2026-07-01 08:10:00,88,85.0,3.0
